# Telecom X - Parte 2: Prevendo Churn

## Contexto
Após a etapa de análise exploratória, este caderno apresenta a fase de Machine Learning para prever evasão de clientes.

## Objetivo
Construir e avaliar modelos de classificação para estimar a chance de churn e transformar resultados técnicos em recomendações estratégicas para retenção.

## Plano do caderno
1. Carregar base tratada em CSV
2. Preparar dados para modelagem
3. Analisar correlação e relações com a evasão
4. Treinar dois modelos (Regressão Logística e Random Forest)
5. Avaliar métricas e overfitting/underfitting
6. Interpretar variáveis mais relevantes
7. Concluir com foco em estratégia de negócio

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 6)
pd.set_option('display.max_columns', None)

In [ ]:
csv_path = Path('TelecomX_tratado.csv')
if not csv_path.exists():
    raise FileNotFoundError('Arquivo TelecomX_tratado.csv não encontrado. Rode a Parte 1 ou gere o CSV tratado antes.')

df = pd.read_csv(csv_path)
print(f'Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas')
display(df.head(3))
display(df.dtypes.to_frame('tipo_dado'))

## Preparação para modelagem
- Remoção de coluna identificadora (IdCliente)
- Definição da variável alvo (evasão, coluna `Evasao`)
- One-Hot Encoding para variáveis categóricas
- Padronização apenas no modelo sensível à escala (Regressão Logística)

In [ ]:
df_model = df.copy()

if 'IdCliente' in df_model.columns:
    df_model = df_model.drop(columns=['IdCliente'])

df_model['Evasao_bin'] = df_model['Evasao'].map({'Sim': 1, 'Nao': 0})
if df_model['Evasao_bin'].isna().any():
    raise ValueError('A coluna `Evasao` possui valores fora do esperado: Sim/Nao.')

X = df_model.drop(columns=['Evasao', 'Evasao_bin'])
y = df_model['Evasao_bin']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Features numéricas: {len(num_features)} | Features categóricas: {len(cat_features)}')
print('Exemplo de features categóricas:', cat_features[:6])

In [ ]:
classe_pct = y.value_counts(normalize=True).mul(100).rename(index={0: 'Permaneceu', 1: 'Evadido'}).sort_index()
classe_abs = y.value_counts().rename(index={0: 'Permaneceu', 1: 'Evadido'}).sort_index()

balanceamento = pd.DataFrame({'Quantidade': classe_abs, 'Percentual (%)': classe_pct.round(2)})
display(balanceamento)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(balanceamento.index, balanceamento['Quantidade'], color=['#4c72b0', '#dd8452'])
ax[0].set_title('Distribuição de classes - absoluto')
ax[0].set_ylabel('Clientes')

ax[1].pie(balanceamento['Percentual (%)'], labels=balanceamento.index, autopct='%1.1f%%', startangle=90, colors=['#4c72b0', '#dd8452'])
ax[1].set_title('Distribuição de classes - percentual')

plt.tight_layout()
plt.show()

print('Observação: o desbalanceamento é moderado. Neste baseline, usamos stratify no split e métricas além da acurácia.')

## Correlação e relações com evasão

In [ ]:
corr_df = df_model.copy()
for col in corr_df.select_dtypes(include='object').columns:
    corr_df[col] = corr_df[col].astype('category').cat.codes

corr = corr_df.corr(numeric_only=True)
top_corr = corr['Evasao_bin'].drop('Evasao_bin').sort_values(key=np.abs, ascending=False).head(12)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Matriz de correlação (inclui alvo numérico `Evasao_bin`)')
plt.tight_layout()
plt.show()

display(top_corr.to_frame('Correlação com `Evasao_bin`'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df_model, x='Evasao', y='Meses_Contrato', hue='Evasao', legend=False, ax=ax[0])
ax[0].set_title('Meses de contrato x Evasão')

sns.boxplot(data=df_model, x='Evasao', y='Gasto_Total', hue='Evasao', legend=False, ax=ax[1])
ax[1].set_title('Gasto total x Evasão')

plt.tight_layout()
plt.show()

## Split de treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)

## Modelos
### Modelo 1 - Regressão Logística (com padronização)
Modelo linear, interpretável e sensível à escala.

### Modelo 2 - Random Forest (sem padronização)
Modelo de árvores, robusto a não linearidades e não sensível à escala.

In [ ]:
preprocess_lr = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_features),
    ]
)

preprocess_rf = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_features),
    ]
)

modelo_lr = Pipeline([
    ('prep', preprocess_lr),
    ('clf', LogisticRegression(max_iter=2000, random_state=42))
])

modelo_rf = Pipeline([
    ('prep', preprocess_rf),
    ('clf', RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced'))
])

modelo_lr.fit(X_train, y_train)
modelo_rf.fit(X_train, y_train)

In [ ]:
def avaliar_modelo(nome, modelo, X_train, y_train, X_test, y_test):
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    resultados = {
        'Modelo': nome,
        'Acuracia_treino': accuracy_score(y_train, y_pred_train),
        'Acuracia_teste': accuracy_score(y_test, y_pred_test),
        'Precisao': precision_score(y_test, y_pred_test),
        'Recall': recall_score(y_test, y_pred_test),
        'F1-score': f1_score(y_test, y_pred_test),
    }

    cm = confusion_matrix(y_test, y_pred_test)

    print(f'--- {nome} ---')
    print(classification_report(y_test, y_pred_test, target_names=['Permaneceu', 'Evadido']))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Permaneceu', 'Evadido'])
    disp.plot(cmap='Blues')
    plt.title(f'Matriz de confusão - {nome}')
    plt.show()

    return resultados

res_lr = avaliar_modelo('Regressão Logística', modelo_lr, X_train, y_train, X_test, y_test)
res_rf = avaliar_modelo('Random Forest', modelo_rf, X_train, y_train, X_test, y_test)

df_resultados = pd.DataFrame([res_lr, res_rf]).set_index('Modelo')
display(df_resultados.round(4))

In [ ]:
gap = df_resultados['Acuracia_treino'] - df_resultados['Acuracia_teste']
diagnostico = pd.DataFrame({
    'Gap treino-teste': gap.round(4),
    'Leitura': [
        'Maior risco de overfitting' if v > 0.05 else 'Generalização adequada'
        for v in gap
    ]
})
display(diagnostico)

## Interpretação das variáveis mais relevantes

In [ ]:
feature_names_lr = modelo_lr.named_steps['prep'].get_feature_names_out()
coef_lr = modelo_lr.named_steps['clf'].coef_[0]
imp_lr = pd.Series(coef_lr, index=feature_names_lr).sort_values(key=np.abs, ascending=False).head(15)

feature_names_rf = modelo_rf.named_steps['prep'].get_feature_names_out()
imp_rf_vals = modelo_rf.named_steps['clf'].feature_importances_
imp_rf = pd.Series(imp_rf_vals, index=feature_names_rf).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
imp_lr.sort_values().plot(kind='barh', ax=ax[0], color='#4c72b0')
ax[0].set_title('Top 15 coeficientes (abs) - Regressão Logística')

imp_rf.sort_values().plot(kind='barh', ax=ax[1], color='#55a868')
ax[1].set_title('Top 15 importâncias - Random Forest')

plt.tight_layout()
plt.show()

display(imp_lr.to_frame('coef_lr_abs_top'))
display(imp_rf.to_frame('rf_importance_top'))

In [ ]:
melhor_modelo = df_resultados['F1-score'].idxmax()
melhor_f1 = df_resultados.loc[melhor_modelo, 'F1-score']
melhor_recall = df_resultados.loc[melhor_modelo, 'Recall']

top_lr = imp_lr.head(5).index.tolist()
top_rf = imp_rf.head(5).index.tolist()

resumo_md = f"""
## Relatório executivo
- Melhor modelo pelo F1-score: **{melhor_modelo}** (F1 = **{melhor_f1:.4f}**, Recall = **{melhor_recall:.4f}**).
- A comparação entre acurácia de treino e teste indica o nível de generalização de cada abordagem.
- Fatores relevantes observados em ambos os modelos concentram-se em tempo de contrato, padrão de gasto e tipo de serviço/contrato.

### Top sinais no modelo linear (Regressão Logística)
{chr(10).join([f'- `{item}`' for item in top_lr])}

### Top sinais no modelo de árvore (Random Forest)
{chr(10).join([f'- `{item}`' for item in top_rf])}

## Recomendações de retenção
1. Criar ações preventivas para clientes em contrato mensal, principalmente nos primeiros meses.
2. Monitorar clientes com aumento de gasto mensal e baixo tempo de permanência.
3. Priorizar ofertas de fidelização para segmentos com maior propensão de churn.
4. Evoluir a solução com tuning de hiperparâmetros e avaliação com validação cruzada.
"""

display(Markdown(resumo_md))

## Conclusão estratégica

1. O experimento comparou modelos lineares e baseados em árvores para prever churn.
2. As métricas de precisão, recall e F1-score devem ser priorizadas junto com acurácia devido ao desbalanceamento.
3. Variáveis relacionadas a tipo de contrato, tempo de permanência e padrão de gastos aparecem como fatores centrais de risco.
4. Para negócio, a recomendação é focar campanhas preventivas em clientes com sinais de risco alto logo nos primeiros meses.
5. Próximos passos: calibração de hiperparâmetros, validação cruzada e teste com técnicas de balanceamento (ex.: SMOTE).